In [ ]:
import os
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from optuna.samplers import TPESampler
import shap
import traceback
from scipy import stats
from typing import Dict, Tuple, Any, Optional

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Suppress Optuna logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_SEED = 2025
np.random.seed(RANDOM_SEED)


class LightGBMGlucosePredictor:
    """LightGBM Glucose Predictor - Bayesian Optimization Version"""

    def __init__(self, random_state: int = RANDOM_SEED):
        self.random_state = random_state
        self.model_30min = None
        self.model_60min = None
        self.feature_names = None
        self.best_params_30min = None
        self.best_params_60min = None
        self.results_dir = "results"

        # Create results directory
        os.makedirs(self.results_dir, exist_ok=True)

    def load_data(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Load preprocessed data"""
        logger.info("Loading data...")

        try:
            train_df = pd.read_csv("train_enhanced_processed.csv")
            val_df = pd.read_csv("val_enhanced_processed.csv")
            test_df = pd.read_csv("test_enhanced_processed.csv")

            logger.info(f"Training set: {train_df.shape}")
            logger.info(f"Validation set: {val_df.shape}")
            logger.info(f"Test set: {test_df.shape}")

            # Convert timestamp to datetime (if exists)
            for df in [train_df, val_df, test_df]:
                if "timestamp" in df.columns:
                    df["timestamp"] = pd.to_datetime(df["timestamp"])

            return train_df, val_df, test_df

        except FileNotFoundError as e:
            logger.error(f"Data file not found: {e}")
            raise

    def prepare_features(
        self, df: pd.DataFrame, target: str = "glucose_30min"
    ) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare features and target variable"""
        # Columns to exclude
        exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]

        # Get feature columns
        feature_cols = [col for col in df.columns if col not in exclude_cols]

        X = df[feature_cols].copy()
        y = df[target].copy()

        # Remove rows containing NaN
        valid_mask = ~(X.isna().any(axis=1) | y.isna())
        X = X[valid_mask]
        y = y[valid_mask]

        # Save feature names (first call)
        if self.feature_names is None:
            self.feature_names = feature_cols

        logger.info(f"Target variable: {target}")
        logger.info(f"Number of features: {len(feature_cols)}")
        logger.info(f"Number of valid samples: {len(X)}")

        return X, y

    def objective(
        self,
        trial: optuna.Trial,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
    ) -> float:
        """Optuna objective function for Bayesian optimization"""

        # Define hyperparameter search space
        params = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "verbosity": -1,
            "random_state": self.random_state,  # 确保使用您的随机种子
            "n_jobs": -1,  # 使用所有可用的CPU核心
            "force_col_wise": True,
            # 超大范围超参数
            "n_estimators": trial.suggest_int(
                "n_estimators", 100, 5000
            ),  # 树的数量：从100到5000
            "max_depth": trial.suggest_int("max_depth", 3, 50),  # 最大深度：从3到50
            "num_leaves": trial.suggest_int(
                "num_leaves", 10, 1000
            ),  # 叶子数量：从10到1000
            "learning_rate": trial.suggest_float(
                "learning_rate", 1e-5, 0.5, log=True
            ),  # 学习率：从0.00001到0.5（对数尺度）
            "min_child_samples": trial.suggest_int(
                "min_child_samples", 1, 200
            ),  # 叶子节点最小样本数：从1到200
            "subsample": trial.suggest_float(
                "subsample", 0.5, 1.0
            ),  # 样本采样比例：从0.5到1.0
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.5, 1.0
            ),  # 特征采样比例：从0.5到1.0
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-5, 100.0, log=True
            ),  # L1正则化：从0.00001到100（对数尺度）
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-5, 100.0, log=True
            ),  # L2正则化：从0.00001到100（对数尺度）
            "min_split_gain": trial.suggest_float(
                "min_split_gain", 0.0, 1.0
            ),  # 分裂最小增益：从0.0到1.0
            # 可选：添加其他参数以进一步扩展搜索空间
            "bagging_freq": trial.suggest_int(
                "bagging_freq", 0, 10
            ),  # 袋频率：从0（禁用）到10
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 1e-3, 10.0, log=True
            ),  # 叶子节点最小海森值：从0.001到10（对数尺度）
            "max_bin": trial.suggest_int("max_bin", 10, 500),  # 最大分箱数：从10到500
        }

        # Create model
        model = lgb.LGBMRegressor(**params)

        # Train with early stopping
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        # Predict on validation set
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        return rmse

    def tune_hyperparameters(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        n_trials: int = 200,
    ) -> Tuple[lgb.LGBMRegressor, Dict[str, Any]]:
        """Tune hyperparameters using Bayesian optimization (Optuna)"""
        logger.info("=" * 60)
        logger.info("Starting Bayesian hyperparameter optimization...")
        logger.info(f"Number of optimization trials: {n_trials}")
        logger.info("=" * 60)

        # Create Optuna study
        study = optuna.create_study(
            direction="minimize",
            sampler=TPESampler(seed=self.random_state),
            study_name="LightGBM_Glucose_Prediction",
        )

        # Run optimization
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )

        # Get best parameters
        best_params = study.best_params
        best_rmse = study.best_value

        logger.info("\nBest parameters found:")
        for param, value in best_params.items():
            logger.info(f"  {param}: {value}")
        logger.info(f"\nBest validation RMSE: {best_rmse:.4f} mg/dL")

        # Train final model with best parameters
        final_params = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "verbosity": -1,
            "random_state": self.random_state,
            "n_jobs": -1,
            "force_col_wise": True,
            **best_params,
        }

        best_model = lgb.LGBMRegressor(**final_params)
        best_model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        # Save optimization history
        optimization_history = pd.DataFrame(
            {
                "trial": range(len(study.trials)),
                "value": [trial.value for trial in study.trials],
            }
        )

        return best_model, best_params, optimization_history

    def train_models(
        self, train_df: pd.DataFrame, val_df: pd.DataFrame, n_trials: int = 200
    ) -> None:
        """Train 30-minute and 60-minute prediction models"""

        # Train 30-minute prediction model
        logger.info("\n" + "=" * 60)
        logger.info("Training 30-minute glucose prediction model")
        logger.info("=" * 60)

        X_train_30, y_train_30 = self.prepare_features(train_df, "glucose_30min")
        X_val_30, y_val_30 = self.prepare_features(val_df, "glucose_30min")

        self.model_30min, self.best_params_30min, history_30 = (
            self.tune_hyperparameters(
                X_train_30, y_train_30, X_val_30, y_val_30, n_trials=n_trials
            )
        )

        # Save optimization history
        history_30.to_csv(
            f"{self.results_dir}/optimization_history_30min.csv", index=False
        )

        # Validation set evaluation
        val_pred_30 = self.model_30min.predict(X_val_30)
        val_rmse_30 = np.sqrt(mean_squared_error(y_val_30, val_pred_30))
        val_mae_30 = mean_absolute_error(y_val_30, val_pred_30)
        val_r2_30 = r2_score(y_val_30, val_pred_30)

        logger.info("\n30-minute model validation set performance:")
        logger.info(f"  RMSE: {val_rmse_30:.4f} mg/dL")
        logger.info(f"  MAE: {val_mae_30:.4f} mg/dL")
        logger.info(f"  R²: {val_r2_30:.4f}")

        # Train 60-minute prediction model
        logger.info("\n" + "=" * 60)
        logger.info("Training 60-minute glucose prediction model")
        logger.info("=" * 60)

        X_train_60, y_train_60 = self.prepare_features(train_df, "glucose_60min")
        X_val_60, y_val_60 = self.prepare_features(val_df, "glucose_60min")

        self.model_60min, self.best_params_60min, history_60 = (
            self.tune_hyperparameters(
                X_train_60, y_train_60, X_val_60, y_val_60, n_trials=n_trials
            )
        )

        # Save optimization history
        history_60.to_csv(
            f"{self.results_dir}/optimization_history_60min.csv", index=False
        )

        # Validation set evaluation
        val_pred_60 = self.model_60min.predict(X_val_60)
        val_rmse_60 = np.sqrt(mean_squared_error(y_val_60, val_pred_60))
        val_mae_60 = mean_absolute_error(y_val_60, val_pred_60)
        val_r2_60 = r2_score(y_val_60, val_pred_60)

        logger.info("\n60-minute model validation set performance:")
        logger.info(f"  RMSE: {val_rmse_60:.4f} mg/dL")
        logger.info(f"  MAE: {val_mae_60:.4f} mg/dL")
        logger.info(f"  R²: {val_r2_60:.4f}")

        # Save model parameters
        params_df = pd.DataFrame(
            {"30min": self.best_params_30min, "60min": self.best_params_60min}
        )
        params_df.to_csv(f"{self.results_dir}/best_hyperparameters.csv")
        logger.info(
            f"\nBest hyperparameters saved to {self.results_dir}/best_hyperparameters.csv"
        )

        # Plot optimization history
        self.plot_optimization_history(history_30, history_60)

    def plot_optimization_history(
        self, history_30: pd.DataFrame, history_60: pd.DataFrame
    ) -> None:
        """Plot Bayesian optimization history"""
        logger.info("\nGenerating optimization history plots...")

        fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

        # 30-minute model
        axes[0].plot(history_30["trial"], history_30["value"], "b-", alpha=0.6)
        axes[0].plot(
            history_30["trial"],
            history_30["value"].cummin(),
            "r-",
            linewidth=2,
            label="Best RMSE",
        )
        axes[0].set_xlabel("Trial", fontsize=12)
        axes[0].set_ylabel("Validation RMSE (mg/dL)", fontsize=12)
        axes[0].set_title(
            "30-min Model - Bayesian Optimization History",
            fontsize=14,
            fontweight="bold",
        )
        axes[0].legend(fontsize=10)
        axes[0].grid(True, alpha=0.3)

        # 60-minute model
        axes[1].plot(history_60["trial"], history_60["value"], "b-", alpha=0.6)
        axes[1].plot(
            history_60["trial"],
            history_60["value"].cummin(),
            "r-",
            linewidth=2,
            label="Best RMSE",
        )
        axes[1].set_xlabel("Trial", fontsize=12)
        axes[1].set_ylabel("Validation RMSE (mg/dL)", fontsize=12)
        axes[1].set_title(
            "60-min Model - Bayesian Optimization History",
            fontsize=14,
            fontweight="bold",
        )
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(
            f"{self.results_dir}/optimization_history.png", dpi=300, bbox_inches="tight"
        )
        plt.show()
        plt.close()

        logger.info("Optimization history plots saved")

    def evaluate_on_test(self, test_df: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
        """Evaluate on test set"""
        logger.info("\n" + "=" * 60)
        logger.info("Test set evaluation")
        logger.info("=" * 60)

        results = {}

        # 30-minute prediction
        X_test_30, y_test_30 = self.prepare_features(test_df, "glucose_30min")
        pred_30 = self.model_30min.predict(X_test_30)

        rmse_30 = np.sqrt(mean_squared_error(y_test_30, pred_30))
        mae_30 = mean_absolute_error(y_test_30, pred_30)
        r2_30 = r2_score(y_test_30, pred_30)

        logger.info("\n30-minute prediction:")
        logger.info(f"  RMSE: {rmse_30:.4f} mg/dL")
        logger.info(f"  MAE: {mae_30:.4f} mg/dL")
        logger.info(f"  R²: {r2_30:.4f}")

        results["30min"] = {
            "y_true": y_test_30,
            "y_pred": pred_30,
            "rmse": rmse_30,
            "mae": mae_30,
            "r2": r2_30,
            "X": X_test_30,
        }

        # 60-minute prediction
        X_test_60, y_test_60 = self.prepare_features(test_df, "glucose_60min")
        pred_60 = self.model_60min.predict(X_test_60)

        rmse_60 = np.sqrt(mean_squared_error(y_test_60, pred_60))
        mae_60 = mean_absolute_error(y_test_60, pred_60)
        r2_60 = r2_score(y_test_60, pred_60)

        logger.info("\n60-minute prediction:")
        logger.info(f"  RMSE: {rmse_60:.4f} mg/dL")
        logger.info(f"  MAE: {mae_60:.4f} mg/dL")
        logger.info(f"  R²: {r2_60:.4f}")

        results["60min"] = {
            "y_true": y_test_60,
            "y_pred": pred_60,
            "rmse": rmse_60,
            "mae": mae_60,
            "r2": r2_60,
            "X": X_test_60,
        }

        # Save prediction results
        for horizon in ["30min", "60min"]:
            result_df = pd.DataFrame(
                {
                    "y_true": results[horizon]["y_true"].values,
                    "y_pred": results[horizon]["y_pred"],
                }
            )
            result_df.to_csv(
                f"{self.results_dir}/predictions_{horizon}.csv", index=False
            )

        logger.info(f"\nPrediction results saved to {self.results_dir}/")

        return results

    def plot_predictions(self, results: Dict[str, Dict[str, Any]]) -> None:
        """Plot prediction comparison (4-in-1 plot)"""
        logger.info("\nGenerating prediction comparison plots...")

        for horizon, data in results.items():
            y_true = data["y_true"].values
            y_pred = data["y_pred"]

            # Select subset of data points for visualization (avoid overcrowding)
            plot_samples = min(500, len(y_true))
            indices = np.linspace(0, len(y_true) - 1, plot_samples, dtype=int)

            fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)

            # 1. Time series comparison
            axes[0, 0].plot(
                indices,
                y_true[indices],
                "b-",
                label="Ground Truth",
                alpha=0.7,
                linewidth=1.5,
            )
            axes[0, 0].plot(
                indices,
                y_pred[indices],
                "r--",
                label="Prediction",
                alpha=0.7,
                linewidth=1.5,
            )
            axes[0, 0].fill_between(
                indices, y_true[indices], y_pred[indices], alpha=0.2, color="gray"
            )
            axes[0, 0].set_xlabel("Sample Index", fontsize=11)
            axes[0, 0].set_ylabel("Glucose Level (mg/dL)", fontsize=11)
            axes[0, 0].set_title(
                f"{horizon} Glucose Prediction - Time Series Comparison",
                fontsize=13,
                fontweight="bold",
            )
            axes[0, 0].legend(fontsize=10)
            axes[0, 0].grid(True, alpha=0.3)

            # 2. Scatter plot
            axes[0, 1].scatter(y_true, y_pred, alpha=0.3, s=10, c="steelblue")

            # Add perfect prediction line
            min_val = min(y_true.min(), y_pred.min())
            max_val = max(y_true.max(), y_pred.max())
            axes[0, 1].plot(
                [min_val, max_val],
                [min_val, max_val],
                "r--",
                linewidth=2,
                label="Perfect Prediction",
                alpha=0.8,
            )

            axes[0, 1].set_xlabel("True Glucose Level (mg/dL)", fontsize=11)
            axes[0, 1].set_ylabel("Predicted Glucose Level (mg/dL)", fontsize=11)
            axes[0, 1].set_title(
                f"{horizon} Glucose Prediction - Scatter Plot",
                fontsize=13,
                fontweight="bold",
            )
            axes[0, 1].legend(fontsize=10)
            axes[0, 1].grid(True, alpha=0.3)

            # Add performance metrics
            textstr = f"RMSE: {data['rmse']:.2f} mg/dL\nMAE: {data['mae']:.2f} mg/dL\nR²: {data['r2']:.4f}"
            axes[0, 1].text(
                0.05,
                0.95,
                textstr,
                transform=axes[0, 1].transAxes,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7),
                fontsize=10,
            )

            # 3. Residual plot
            residuals = y_true - y_pred
            axes[1, 0].scatter(y_pred, residuals, alpha=0.3, s=10, c="coral")
            axes[1, 0].axhline(y=0, color="r", linestyle="--", linewidth=2)
            axes[1, 0].set_xlabel("Predicted Glucose Level (mg/dL)", fontsize=11)
            axes[1, 0].set_ylabel("Residual (mg/dL)", fontsize=11)
            axes[1, 0].set_title(
                f"{horizon} Glucose Prediction - Residual Plot",
                fontsize=13,
                fontweight="bold",
            )
            axes[1, 0].grid(True, alpha=0.3)

            # Add residual statistics
            residual_mean = np.mean(residuals)
            residual_std = np.std(residuals)
            axes[1, 0].text(
                0.05,
                0.95,
                f"Mean: {residual_mean:.2f}\nStd Dev: {residual_std:.2f}",
                transform=axes[1, 0].transAxes,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
                fontsize=10,
            )

            # 4. Residual distribution
            axes[1, 1].hist(
                residuals, bins=50, alpha=0.7, color="blue", edgecolor="black"
            )
            axes[1, 1].axvline(
                x=0, color="r", linestyle="--", linewidth=2, label="Zero Residual"
            )

            # Add normal distribution fit curve
            mu, sigma = stats.norm.fit(residuals)
            x = np.linspace(residuals.min(), residuals.max(), 100)
            p = stats.norm.pdf(x, mu, sigma)
            ax2 = axes[1, 1].twinx()
            ax2.plot(x, p, "r-", linewidth=2, label="Normal Distribution Fit")
            ax2.set_ylabel("Probability Density", fontsize=10)

            axes[1, 1].set_xlabel("Residual (mg/dL)", fontsize=11)
            axes[1, 1].set_ylabel("Frequency", fontsize=11)
            axes[1, 1].set_title(
                f"{horizon} Glucose Prediction - Residual Distribution",
                fontsize=13,
                fontweight="bold",
            )
            axes[1, 1].legend(loc="upper left", fontsize=9)
            ax2.legend(loc="upper right", fontsize=9)
            axes[1, 1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(
                f"{self.results_dir}/prediction_comparison_{horizon}.png",
                dpi=300,
                bbox_inches="tight",
            )
            plt.show()
            plt.close()

            logger.info(f"{horizon} prediction curves saved")

    def plot_time_series_by_patient(
        self,
        results: Dict[str, Dict[str, Any]],
        test_df: pd.DataFrame,
        num_patients: int = 2,
    ) -> None:
        """Plot time series by patient (if timestamp and patient_id are available)"""

        if "timestamp" not in test_df.columns or "patient_id" not in test_df.columns:
            logger.warning(
                "Missing timestamp or patient_id columns, skipping patient-level time series plots"
            )
            return

        logger.info("\nGenerating patient-level time series plots...")

        patient_ids = test_df["patient_id"].unique()[:num_patients]

        for horizon, data in results.items():
            for patient_id in patient_ids:
                try:
                    patient_mask = test_df["patient_id"] == patient_id
                    patient_data = test_df[patient_mask].reset_index(drop=True)

                    # Get predictions and ground truth for this patient
                    target_col = (
                        "glucose_30min" if horizon == "30min" else "glucose_60min"
                    )

                    if len(patient_data) < 10:
                        continue

                    y_true_patient = patient_data[target_col].dropna().values
                    timestamps = (
                        patient_data["timestamp"].iloc[: len(y_true_patient)].values
                    )

                    # Get corresponding predictions
                    patient_indices = test_df[patient_mask].index
                    y_pred_patient = []
                    for idx in patient_indices[: len(y_true_patient)]:
                        if idx in data["y_true"].index:
                            pred_idx = data["y_true"].index.get_loc(idx)
                            y_pred_patient.append(data["y_pred"][pred_idx])

                    y_pred_patient = np.array(y_pred_patient)

                    if len(y_pred_patient) != len(y_true_patient):
                        continue

                    # Plot
                    fig, ax = plt.subplots(figsize=(16, 6), dpi=300)

                    ax.plot(
                        timestamps,
                        y_true_patient,
                        "b-",
                        label="Ground Truth",
                        linewidth=2,
                        alpha=0.7,
                    )
                    ax.plot(
                        timestamps,
                        y_pred_patient,
                        "r--",
                        label="Prediction",
                        linewidth=2,
                        alpha=0.7,
                    )
                    ax.fill_between(
                        timestamps,
                        y_true_patient,
                        y_pred_patient,
                        alpha=0.2,
                        color="gray",
                    )

                    # Add target range
                    ax.axhspan(
                        70,
                        180,
                        alpha=0.1,
                        color="green",
                        label="Target Range (70-180 mg/dL)",
                    )

                    ax.set_xlabel("Time", fontsize=12)
                    ax.set_ylabel("Glucose Level (mg/dL)", fontsize=12)
                    ax.set_title(
                        f"{horizon} Glucose Prediction - Patient {patient_id} Time Series",
                        fontsize=14,
                        fontweight="bold",
                        pad=20,
                    )
                    ax.legend(fontsize=10, loc="best")
                    ax.grid(True, alpha=0.3)

                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.savefig(
                        f"{self.results_dir}/time_series_{horizon}_patient_{patient_id}.png",
                        dpi=300,
                        bbox_inches="tight",
                    )
                    plt.show()
                    plt.close()

                except Exception as e:
                    logger.warning(
                        f"Failed to plot {horizon} time series for patient {patient_id}: {e}"
                    )
                    continue

    def shap_analysis(
        self, results: Dict[str, Dict[str, Any]], max_samples: int = 100
    ) -> None:
        """SHAP interpretability analysis"""
        logger.info("\n" + "=" * 60)
        logger.info("SHAP Interpretability Analysis")
        logger.info("=" * 60)

        for horizon, data in results.items():
            logger.info(f"\nAnalyzing {horizon} prediction model...")

            model = self.model_30min if horizon == "30min" else self.model_60min
            X = data["X"]

            # Limit sample size to improve computational efficiency
            if len(X) > max_samples:
                sample_indices = np.random.choice(len(X), max_samples, replace=False)
                X_sample = X.iloc[sample_indices]
            else:
                X_sample = X

            try:
                # Create SHAP explainer
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_sample)

                # 1. Summary Plot
                plt.figure(figsize=(12, 8), dpi=300)
                shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Summary Plot",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_summary_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 2. Bar Plot (Feature Importance)
                plt.figure(figsize=(10, 8), dpi=300)
                shap.summary_plot(
                    shap_values, X_sample, plot_type="bar", show=False, max_display=20
                )
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Feature Importance",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_importance_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 3. Waterfall Plot (first sample)
                plt.figure(figsize=(12, 8), dpi=300)
                shap.plots.waterfall(
                    shap.Explanation(
                        values=shap_values[0],
                        base_values=explainer.expected_value,
                        data=X_sample.iloc[0],
                        feature_names=X_sample.columns.tolist(),
                    ),
                    show=False,
                    max_display=15,
                )
                plt.title(
                    f"{horizon} Glucose Prediction - SHAP Waterfall Plot (Sample 1)",
                    pad=20,
                    fontsize=14,
                    fontweight="bold",
                )
                plt.tight_layout()
                plt.savefig(
                    f"{self.results_dir}/shap_waterfall_{horizon}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                # 4. Calculate and save feature importance
                feature_importance = pd.DataFrame(
                    {
                        "feature": X_sample.columns,
                        "importance": np.abs(shap_values).mean(axis=0),
                    }
                ).sort_values("importance", ascending=False)

                logger.info(f"\n{horizon} prediction - Top 15 important features:")
                for idx, row in feature_importance.head(15).iterrows():
                    logger.info(f"  {row['feature']}: {row['importance']:.4f}")

                # Save feature importance
                feature_importance.to_csv(
                    f"{self.results_dir}/feature_importance_{horizon}.csv", index=False
                )

                logger.info(f"\n{horizon} SHAP analysis completed")

            except Exception as e:
                logger.error(f"SHAP analysis failed ({horizon}): {e}")
                continue


def run_complete_pipeline(
    n_trials: int = 200, shap_samples: int = 100
) -> Tuple[LightGBMGlucosePredictor, Dict]:
    """Run complete prediction pipeline"""

    logger.info("=" * 60)
    logger.info("LightGBM Glucose Prediction Pipeline - Bayesian Optimization Version")
    logger.info("=" * 60)

    try:
        # Initialize predictor
        predictor = LightGBMGlucosePredictor(random_state=RANDOM_SEED)

        # Load data
        train_df, val_df, test_df = predictor.load_data()

        # Train models
        predictor.train_models(train_df, val_df, n_trials=n_trials)

        # Test set evaluation
        results = predictor.evaluate_on_test(test_df)

        # Plot prediction curves (4-in-1 plot)
        predictor.plot_predictions(results)

        # Plot patient-level time series (if available in data)
        predictor.plot_time_series_by_patient(results, test_df, num_patients=2)

        # SHAP interpretability analysis
        predictor.shap_analysis(results, max_samples=shap_samples)

        # Generate summary report
        generate_summary_report(predictor, results)

        logger.info("\n" + "=" * 60)
        logger.info("Pipeline execution completed!")
        logger.info(f"All results saved to: {predictor.results_dir}/")
        logger.info("=" * 60)

        return predictor, results

    except Exception as e:
        logger.error(f"Pipeline execution failed: {e}")
        raise


def generate_summary_report(predictor: LightGBMGlucosePredictor, results: Dict) -> None:
    """Generate summary report"""
    report_path = f"{predictor.results_dir}/summary_report.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("LightGBM Glucose Prediction Model - Summary Report\n")
        f.write("(Bayesian Optimization Version)\n")
        f.write("=" * 60 + "\n\n")

        f.write("1. Model Performance\n")
        f.write("-" * 40 + "\n")
        for horizon, data in results.items():
            f.write(f"\n{horizon} prediction:\n")
            f.write(f"  RMSE: {data['rmse']:.4f} mg/dL\n")
            f.write(f"  MAE:  {data['mae']:.4f} mg/dL\n")
            f.write(f"  R²:   {data['r2']:.4f}\n")

        f.write("\n\n2. Best Hyperparameters (Bayesian Optimization)\n")
        f.write("-" * 40 + "\n")

        f.write("\n30-minute model:\n")
        for param, value in predictor.best_params_30min.items():
            f.write(f"  {param}: {value}\n")

        f.write("\n60-minute model:\n")
        for param, value in predictor.best_params_60min.items():
            f.write(f"  {param}: {value}\n")

        f.write("\n\n3. Number of Features\n")
        f.write("-" * 40 + "\n")
        f.write(f"Total features: {len(predictor.feature_names)}\n")

        f.write("\n\n4. Output Files List\n")
        f.write("-" * 40 + "\n")
        f.write("  - best_hyperparameters.csv (Best hyperparameters)\n")
        f.write("  - optimization_history_30min.csv (30-min optimization history)\n")
        f.write("  - optimization_history_60min.csv (60-min optimization history)\n")
        f.write("  - optimization_history.png (Optimization convergence plot)\n")
        f.write("  - predictions_30min.csv (30-minute prediction results)\n")
        f.write("  - predictions_60min.csv (60-minute prediction results)\n")
        f.write("  - feature_importance_30min.csv (30-minute feature importance)\n")
        f.write("  - feature_importance_60min.csv (60-minute feature importance)\n")
        f.write(
            "  - prediction_comparison_30min.png (30-minute prediction comparison plot)\n"
        )
        f.write(
            "  - prediction_comparison_60min.png (60-minute prediction comparison plot)\n"
        )
        f.write("  - shap_summary_30min.png (30-minute SHAP summary)\n")
        f.write("  - shap_summary_60min.png (60-minute SHAP summary)\n")
        f.write("  - shap_importance_30min.png (30-minute SHAP importance)\n")
        f.write("  - shap_importance_60min.png (60-minute SHAP importance)\n")
        f.write("  - shap_waterfall_30min.png (30-minute SHAP waterfall)\n")
        f.write("  - shap_waterfall_60min.png (60-minute SHAP waterfall)\n")

        f.write("\n\n5. Model Interpretation\n")
        f.write("-" * 40 + "\n")
        f.write("RMSE (Root Mean Square Error): Lower is better\n")
        f.write("MAE (Mean Absolute Error): Lower is better\n")
        f.write("R² (R-squared): Range 0-1, closer to 1 is better\n")
        f.write("\nClinical glucose reference ranges:\n")
        f.write("  - Normal range: 70-180 mg/dL\n")
        f.write("  - Hypoglycemia: < 70 mg/dL\n")
        f.write("  - Hyperglycemia: > 180 mg/dL\n")

        f.write("\n\n6. Optimization Method\n")
        f.write("-" * 40 + "\n")
        f.write("Method: Bayesian Optimization (Optuna TPESampler)\n")
        f.write("Advantage: More efficient than random search, automatically\n")
        f.write("           balances exploration and exploitation to find\n")
        f.write("           optimal hyperparameters faster.\n")

        f.write("\n" + "=" * 60 + "\n")

    logger.info(f"Summary report saved to: {report_path}")


if __name__ == "__main__":
    """
    Usage instructions:
    1. Ensure data files are in the current directory:
       - train_enhanced_processed.csv
       - val_enhanced_processed.csv
       - test_enhanced_processed.csv

    2. Adjust parameters:
       - n_trials: Number of Bayesian optimization trials (default 50, can increase to 100+)
       - shap_samples: SHAP analysis sample size (default 100)

    3. Run the code:
       python script_name.py

    4. Results are saved in the results/ directory

    Note: Bayesian optimization (Optuna) is more efficient than random search,
          typically achieving better results with fewer iterations.
    """

    # Set parameters
    N_TRIALS = 500  # Bayesian optimization trials, can increase for better results
    SHAP_SAMPLES = 100  # SHAP analysis sample size

    # Run complete pipeline
    try:
        predictor, results = run_complete_pipeline(
            n_trials=N_TRIALS, shap_samples=SHAP_SAMPLES
        )

        # Print final summary
        print("\n" + "=" * 60)
        print("Execution completed! Main results:")
        print("=" * 60)
        print("\nTest set performance:")
        for horizon, data in results.items():
            print(f"\n{horizon}:")
            print(f"  RMSE: {data['rmse']:.4f} mg/dL")
            print(f"  MAE:  {data['mae']:.4f} mg/dL")
            print(f"  R²:   {data['r2']:.4f}")

        print(f"\nAll results saved to: {predictor.results_dir}/")
        print("\nMain files:")
        print("  - summary_report.txt (Summary report)")
        print("  - optimization_history.png (Bayesian optimization convergence)")
        print("  - prediction_comparison_*.png (Prediction comparison plots)")
        print("  - shap_*.png (SHAP interpretability plots)")
        print("  - predictions_*.csv (Prediction results)")
        print("  - feature_importance_*.csv (Feature importance)")

    except Exception as e:
        logger.error(f"Execution failed: {e}")
        import traceback

        traceback.print_exc()

In [ ]:
def plot_predictions(
    results: Dict[str, Dict[str, Any]], sample_points: int = 600
) -> None:
    """
    Plot prediction comparison (4-in-1 plot)

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        包含预测结果的字典
    sample_points : int
        要绘制的前N个时间点数量(默认300)
    """
    logger.info("\nGenerating prediction comparison plots...")

    for horizon, data in results.items():
        y_true = data["y_true"]
        y_pred = data["y_pred"]

        # 直接取前 sample_points 个点,而不是采样
        plot_samples = min(sample_points, len(y_true))
        y_true_plot = y_true[:plot_samples]
        y_pred_plot = y_pred[:plot_samples]
        indices = np.arange(plot_samples)  # 0, 1, 2, ..., plot_samples-1

        fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)

        # 1. Time series comparison
        axes[0, 0].plot(
            indices,
            y_true_plot,
            "b-",
            label="Ground Truth",
            alpha=0.7,
            linewidth=1.5,
        )
        axes[0, 0].plot(
            indices,
            y_pred_plot,
            "r--",
            label="Prediction",
            alpha=0.7,
            linewidth=1.5,
        )
        axes[0, 0].fill_between(
            indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray"
        )
        axes[0, 0].set_xlabel("Time Point (5-min intervals)", fontsize=11)
        axes[0, 0].set_ylabel("Glucose Level (mg/dL)", fontsize=11)
        axes[0, 0].set_title(
            f"{horizon} Glucose Prediction - Time Series Comparison (First {plot_samples} points)",
            fontsize=13,
            fontweight="bold",
        )
        axes[0, 0].legend(fontsize=10)
        axes[0, 0].grid(True, alpha=0.3)

        # 2. Scatter plot (使用全部数据点,保持原有逻辑)
        axes[0, 1].scatter(y_true, y_pred, alpha=0.3, s=10, c="steelblue")

        # Add perfect prediction line
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        axes[0, 1].plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Perfect Prediction",
            alpha=0.8,
        )

        axes[0, 1].set_xlabel("True Glucose Level (mg/dL)", fontsize=11)
        axes[0, 1].set_ylabel("Predicted Glucose Level (mg/dL)", fontsize=11)
        axes[0, 1].set_title(
            f"{horizon} Glucose Prediction - Scatter Plot",
            fontsize=13,
            fontweight="bold",
        )
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(True, alpha=0.3)

        # Add performance metrics
        textstr = f"RMSE: {data['rmse']:.2f} mg/dL\nMAE: {data['mae']:.2f} mg/dL\nR²: {data['r2']:.4f}"
        axes[0, 1].text(
            0.05,
            0.95,
            textstr,
            transform=axes[0, 1].transAxes,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7),
            fontsize=10,
        )

        # 3. Residual plot (使用全部数据点)
        residuals = y_true - y_pred
        axes[1, 0].scatter(y_pred, residuals, alpha=0.3, s=10, c="coral")
        axes[1, 0].axhline(y=0, color="r", linestyle="--", linewidth=2)
        axes[1, 0].set_xlabel("Predicted Glucose Level (mg/dL)", fontsize=11)
        axes[1, 0].set_ylabel("Residual (mg/dL)", fontsize=11)
        axes[1, 0].set_title(
            f"{horizon} Glucose Prediction - Residual Plot",
            fontsize=13,
            fontweight="bold",
        )
        axes[1, 0].grid(True, alpha=0.3)

        # Add residual statistics
        residual_mean = np.mean(residuals)
        residual_std = np.std(residuals)
        axes[1, 0].text(
            0.05,
            0.95,
            f"Mean: {residual_mean:.2f}\nStd Dev: {residual_std:.2f}",
            transform=axes[1, 0].transAxes,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
            fontsize=10,
        )

        # 4. Residual distribution (使用全部数据点)
        axes[1, 1].hist(residuals, bins=50, alpha=0.7, color="blue", edgecolor="black")
        axes[1, 1].axvline(
            x=0, color="r", linestyle="--", linewidth=2, label="Zero Residual"
        )

        # Add normal distribution fit curve
        mu, sigma = stats.norm.fit(residuals)
        x = np.linspace(residuals.min(), residuals.max(), 100)
        p = stats.norm.pdf(x, mu, sigma)
        ax2 = axes[1, 1].twinx()
        ax2.plot(x, p, "r-", linewidth=2, label="Normal Distribution Fit")
        ax2.set_ylabel("Probability Density", fontsize=10)

        axes[1, 1].set_xlabel("Residual (mg/dL)", fontsize=11)
        axes[1, 1].set_ylabel("Frequency", fontsize=11)
        axes[1, 1].set_title(
            f"{horizon} Glucose Prediction - Residual Distribution",
            fontsize=13,
            fontweight="bold",
        )
        axes[1, 1].legend(loc="upper left", fontsize=9)
        ax2.legend(loc="upper right", fontsize=9)
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        plt.close()

        logger.info(f"{horizon} prediction curves saved")


plot_predictions(results)

In [ ]:
def plot_predictions(
    results: Dict[str, Dict[str, Any]], sample_points: int = 600
) -> None:
    """
    绘制预测曲线对比图

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        包含预测结果的字典
    sample_points : int
        要绘制的前N个时间点数量(默认300)
    """
    logger.info("\nGenerating prediction comparison plots...")

    for horizon, data in results.items():
        y_true = data["y_true"]
        y_pred = data["y_pred"]

        # 直接取前 sample_points 个点
        plot_samples = min(sample_points, len(y_true))
        y_true_plot = y_true[:plot_samples]
        y_pred_plot = y_pred[:plot_samples]
        indices = np.arange(plot_samples)

        # 创建单个图表
        plt.figure(figsize=(18, 6), dpi=300)

        # 绘制真实值
        plt.plot(
            indices,
            y_true_plot,
            label="True Glucose (mg/dL)",
            color="black",
            linewidth=2,
            alpha=0.8,
        )

        # 绘制预测值
        plt.plot(
            indices,
            y_pred_plot,
            label="Predicted Glucose (mg/dL)",
            color="red",
            linestyle="--",
            linewidth=1.5,
            alpha=0.8,
        )

        # 填充区域
        plt.fill_between(indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray")

        plt.title(
            f"Glucose Prediction Comparison on Test Set - {horizon} (First {plot_samples} points)",
            fontsize=14,
            fontweight="bold",
        )
        plt.xlabel("Time (5-min intervals)", fontsize=12)
        plt.ylabel("Glucose (mg/dL)", fontsize=12)
        plt.legend(loc="upper right", fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.margins(x=0.02, y=0.1)
        plt.tight_layout()
        plt.show()
        plt.close()

        logger.info(f"{horizon} prediction curve saved")


plot_predictions(results)

In [ ]:
for k, v in results["30min"].items():
    print(k)

In [ ]:
results["30min"]["y_true"]

In [ ]:
results["30min"]["y_pred"]

In [ ]:
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import logging
import traceback

logger = logging.getLogger(__name__)


def plot_time_series_by_patient_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    num_patients: int = 6,
    results_dir: str = "results",
) -> None:
    """
    绘制每个患者的血糖时间序列图(独立函数版本)

    Parameters:
    -----------
    results : Dict[str, Dict[str, Any]]
        预测结果字典,包含'30min'和'60min'键
    test_df : pd.DataFrame
        测试数据集
    num_patients : int
        绘制的患者数量
    results_dir : str
        保存结果的目录
    """
    if "timestamp" not in test_df.columns or "patient_id" not in test_df.columns:
        logger.warning("Missing timestamp or patient_id columns")
        return

    logger.info("\nGenerating patient-level time series plots...")

    # 确保时间戳是datetime类型
    test_df = test_df.copy()
    test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

    patient_ids = sorted(test_df["patient_id"].unique())[:num_patients]

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取该患者的数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()

                if len(patient_data) < 10:
                    logger.info(
                        f"Patient {patient_id}: insufficient data (< 10 points)"
                    )
                    continue

                # 移除NaN值
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) == 0:
                    logger.info(
                        f"Patient {patient_id}: no valid data after removing NaN"
                    )
                    continue

                # 重建索引以便对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取对应的预测值
                y_pred_patient = []
                y_true_patient = []
                timestamps = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred_patient.append(data["y_pred"][pred_idx])
                        y_true_patient.append(data["y_true"].iloc[pred_idx])
                        timestamps.append(
                            patient_data[patient_data["index"] == idx][
                                "timestamp"
                            ].iloc[0]
                        )

                if len(y_pred_patient) < 10:
                    logger.info(
                        f"Patient {patient_id}: insufficient matched predictions (< 10 points)"
                    )
                    continue

                y_pred_patient = np.array(y_pred_patient)
                y_true_patient = np.array(y_true_patient)
                timestamps = pd.to_datetime(timestamps)

                # 绘图
                fig, ax = plt.subplots(figsize=(16, 6), dpi=300)

                ax.plot(
                    timestamps,
                    y_true_patient,
                    "b-",
                    label="Ground Truth",
                    linewidth=2,
                    alpha=0.7,
                )
                ax.plot(
                    timestamps,
                    y_pred_patient,
                    "r--",
                    label="Prediction",
                    linewidth=2,
                    alpha=0.7,
                )
                ax.fill_between(
                    timestamps,
                    y_true_patient,
                    y_pred_patient,
                    alpha=0.2,
                    color="gray",
                )

                # 添加目标区域
                ax.axhspan(
                    70,
                    180,
                    alpha=0.1,
                    color="green",
                    label="Target Range (70-180 mg/dL)",
                )

                # 设置x轴格式
                time_range = timestamps.max() - timestamps.min()

                if time_range.days > 7:
                    ax.xaxis.set_major_locator(
                        mdates.DayLocator(interval=max(1, time_range.days // 10))
                    )
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
                elif time_range.days > 1:
                    ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
                else:
                    ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

                plt.xticks(rotation=45)

                ax.set_xlabel("Time", fontsize=12)
                ax.set_ylabel("Glucose Level (mg/dL)", fontsize=12)
                ax.set_title(
                    f"{horizon} Glucose Prediction - Patient {patient_id} Time Series\n"
                    f"(n={len(y_true_patient)} time points)",
                    fontsize=14,
                    fontweight="bold",
                    pad=20,
                )

                ax.legend(fontsize=10, loc="best")
                ax.grid(True, alpha=0.3)

                plt.tight_layout()
                plt.savefig(
                    f"{results_dir}/time_series_{horizon}_patient_{patient_id}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                logger.info(
                    f"Patient {patient_id} ({horizon}): plotted {len(y_true_patient)} time points"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to plot {horizon} for patient {patient_id}: {e}"
                )
                traceback.print_exc()
                continue

In [ ]:
def calculate_patient_metrics_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    results_dir: str = "results",
) -> pd.DataFrame:
    """
    计算每个患者的预测性能指标(RMSE, MAE, R²)

    Parameters:
    -----------
    results : Dict
        预测结果字典
    test_df : pd.DataFrame
        测试数据集
    results_dir : str
        保存结果的目录

    Returns:
    --------
    pd.DataFrame: 包含每个患者性能指标的数据框
    """
    logger.info("\nCalculating per-patient metrics...")

    if "patient_id" not in test_df.columns:
        logger.warning("Missing patient_id column")
        return pd.DataFrame()

    test_df = test_df.copy()
    patient_ids = sorted(test_df["patient_id"].unique())

    all_metrics = []

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取患者数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) < 5:
                    continue

                # 重建索引对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取预测值和真实值
                y_pred = []
                y_true = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred.append(data["y_pred"][pred_idx])
                        y_true.append(data["y_true"].iloc[pred_idx])

                if len(y_pred) < 5:
                    continue

                y_pred = np.array(y_pred)
                y_true = np.array(y_true)

                # 计算指标
                rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                mae = mean_absolute_error(y_true, y_pred)
                r2 = r2_score(y_true, y_pred)

                all_metrics.append(
                    {
                        "patient_id": patient_id,
                        "horizon": horizon,
                        "n_samples": len(y_true),
                        "RMSE": rmse,
                        "MAE": mae,
                        "R2": r2,
                    }
                )

                logger.info(
                    f"Patient {patient_id} ({horizon}): "
                    f"RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}, n={len(y_true)}"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to calculate metrics for patient {patient_id} ({horizon}): {e}"
                )
                continue

    metrics_df = pd.DataFrame(all_metrics)

    # 保存结果
    if len(metrics_df) > 0:
        metrics_df.to_csv(f"{results_dir}/patient_metrics.csv", index=False)
        logger.info(f"\nPatient metrics saved to {results_dir}/patient_metrics.csv")

        # 打印汇总统计
        logger.info("\nSummary statistics across all patients:")
        for horizon in metrics_df["horizon"].unique():
            horizon_data = metrics_df[metrics_df["horizon"] == horizon]
            logger.info(f"\n{horizon}:")
            logger.info(
                f"  RMSE: {horizon_data['RMSE'].mean():.2f} ± {horizon_data['RMSE'].std():.2f}"
            )
            logger.info(
                f"  MAE:  {horizon_data['MAE'].mean():.2f} ± {horizon_data['MAE'].std():.2f}"
            )
            logger.info(
                f"  R²:   {horizon_data['R2'].mean():.4f} ± {horizon_data['R2'].std():.4f}"
            )

    return metrics_df


In [ ]:
def plot_patient_first_n_points_standalone(
    results: Dict[str, Dict[str, Any]],
    test_df: pd.DataFrame,
    n_points: int = 600,
    highlight_ranges: bool = False,
    num_patients: int = None,
    results_dir: str = "results",
) -> None:
    """
    绘制每个患者前N个时间点的真实血糖与预测血糖(独立函数版本)

    Parameters:
    -----------
    results : Dict
        预测结果字典
    test_df : pd.DataFrame
        测试数据
    n_points : int
        绘制的时间点数量(默认600)
    highlight_ranges : bool
        是否标注低血糖和高血糖区域(默认False)
    num_patients : int
        绘制的患者数量(None表示全部)
    results_dir : str
        保存结果的目录
    """
    logger.info(f"\nGenerating first {n_points} points plots for patients...")

    if "patient_id" not in test_df.columns:
        logger.warning("Missing patient_id column")
        return

    test_df = test_df.copy()
    test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])

    patient_ids = sorted(test_df["patient_id"].unique())
    if num_patients is not None:
        patient_ids = patient_ids[:num_patients]

    for horizon, data in results.items():
        target_col = "glucose_30min" if horizon == "30min" else "glucose_60min"

        for patient_id in patient_ids:
            try:
                # 获取患者数据
                patient_mask = test_df["patient_id"] == patient_id
                patient_data = test_df[patient_mask].copy()
                patient_data = patient_data.dropna(subset=[target_col])

                if len(patient_data) == 0:
                    continue

                # 重建索引对齐
                patient_data = patient_data.reset_index(drop=False)
                original_indices = patient_data["index"].values

                # 获取预测值和真实值
                y_pred_list = []
                y_true_list = []
                timestamps_list = []

                for idx in original_indices:
                    if idx in data["y_true"].index:
                        pred_idx = data["y_true"].index.get_loc(idx)
                        y_pred_list.append(data["y_pred"][pred_idx])
                        y_true_list.append(data["y_true"].iloc[pred_idx])
                        timestamps_list.append(
                            patient_data[patient_data["index"] == idx][
                                "timestamp"
                            ].iloc[0]
                        )

                if len(y_pred_list) == 0:
                    continue

                # 截取前n_points个点
                plot_length = min(n_points, len(y_pred_list))
                y_pred_plot = np.array(y_pred_list[:plot_length])
                y_true_plot = np.array(y_true_list[:plot_length])
                timestamps_plot = pd.to_datetime(timestamps_list[:plot_length])
                time_indices = np.arange(plot_length)

                # 计算该患者的指标
                rmse = np.sqrt(mean_squared_error(y_true_plot, y_pred_plot))
                mae = mean_absolute_error(y_true_plot, y_pred_plot)
                r2 = r2_score(y_true_plot, y_pred_plot)

                # 绘图
                fig, ax = plt.subplots(figsize=(18, 6), dpi=300)

                # 如果需要标注血糖范围
                if highlight_ranges:
                    ax.axhspan(
                        0, 70, alpha=0.15, color="red", label="Hypoglycemia (<70 mg/dL)"
                    )
                    ax.axhspan(
                        70,
                        180,
                        alpha=0.1,
                        color="green",
                        label="Target Range (70-180 mg/dL)",
                    )
                    ax.axhspan(
                        180,
                        400,
                        alpha=0.15,
                        color="orange",
                        label="Hyperglycemia (>180 mg/dL)",
                    )

                # 绘制真实值和预测值
                ax.plot(
                    time_indices,
                    y_true_plot,
                    label="True Glucose (mg/dL)",
                    color="black",
                    linewidth=2,
                    alpha=0.8,
                )
                ax.plot(
                    time_indices,
                    y_pred_plot,
                    label="Predicted Glucose (mg/dL)",
                    color="red",
                    linestyle="--",
                    linewidth=1.5,
                    alpha=0.8,
                )

                # 填充区域
                ax.fill_between(
                    time_indices, y_true_plot, y_pred_plot, alpha=0.2, color="gray"
                )

                # 设置标题和标签
                ax.set_title(
                    f"Patient {patient_id} - {horizon} Glucose Prediction (First {plot_length} points)\n"
                    f"RMSE: {rmse:.2f} mg/dL, MAE: {mae:.2f} mg/dL, R²: {r2:.4f}",
                    fontsize=14,
                    fontweight="bold",
                    pad=20,
                )
                ax.set_xlabel("Time Point (5-min intervals)", fontsize=12)
                ax.set_ylabel("Glucose (mg/dL)", fontsize=12)
                ax.legend(loc="best", fontsize=11)
                ax.grid(True, alpha=0.3)
                ax.set_ylim(
                    [
                        min(y_true_plot.min(), y_pred_plot.min()) - 20,
                        max(y_true_plot.max(), y_pred_plot.max()) + 20,
                    ]
                )

                plt.tight_layout()
                plt.savefig(
                    f"{results_dir}/first_{n_points}pts_{horizon}_patient_{patient_id}.png",
                    dpi=300,
                    bbox_inches="tight",
                )
                plt.show()
                plt.close()

                logger.info(
                    f"Patient {patient_id} ({horizon}): "
                    f"plotted {plot_length} points, RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}"
                )

            except Exception as e:
                logger.warning(
                    f"Failed to plot first {n_points} points for patient {patient_id} ({horizon}): {e}"
                )
                traceback.print_exc()
                continue


In [ ]:
# 加载测试数据
test_df = pd.read_csv("test_enhanced_processed.csv")

In [ ]:
# ===== 1. 绘制患者时间序列图 =====
plot_time_series_by_patient_standalone(
    results=results, test_df=test_df, num_patients=6, results_dir="results"
)

In [ ]:
# ===== 2. 计算每个患者的性能指标 =====
patient_metrics = calculate_patient_metrics_standalone(
    results=results, test_df=test_df, results_dir="results"
)

# 查看结果
print("\n患者性能指标:")
print(patient_metrics.head(12))

In [ ]:
# 按患者分组查看
print("\n按患者查看30分钟预测:")
print(patient_metrics[patient_metrics["horizon"] == "30min"])

In [ ]:
# 按患者分组查看
print("\n按患者查看60分钟预测:")
print(patient_metrics[patient_metrics["horizon"] == "60min"])

In [ ]:
# ===== 3. 绘制前600个时间点(不标注血糖范围) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=600,
    highlight_ranges=False,
    num_patients=None,  # 绘制所有患者
    results_dir="results",
)

In [ ]:
# ===== 4. 绘制前600个时间点(标注血糖范围,只绘制前3个患者) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=600,
    highlight_ranges=True,
    num_patients=3,
    results_dir="results",
)

In [ ]:
# ===== 5. 绘制前300个时间点(快速预览) =====
plot_patient_first_n_points_standalone(
    results=results,
    test_df=test_df,
    n_points=300,
    highlight_ranges=False,
    num_patients=5,
    results_dir="results",
)

In [ ]:
import json
import joblib
import pandas as pd
from datetime import datetime
import logging

logger = logging.getLogger(__name__)


def save_models_simple(predictor, results, save_dir="saved_models"):
    """
    简洁版:保存LightGBM模型和预测结果

    Parameters:
    -----------
    predictor : LightGBMGlucosePredictor实例
    results : 测试结果字典
    save_dir : 保存目录
    """
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    logger.info(f"Saving models to {save_dir}...")

    # 1. 保存模型 (joblib格式,推荐)
    joblib.dump(predictor.model_30min, f"{save_dir}/model_30min.pkl")
    joblib.dump(predictor.model_60min, f"{save_dir}/model_60min.pkl")

    # 2. 保存元数据
    metadata = {
        "timestamp": timestamp,
        "feature_names": predictor.feature_names,
        "best_params_30min": predictor.best_params_30min,
        "best_params_60min": predictor.best_params_60min,
    }
    with open(f"{save_dir}/metadata.json", "w") as f:
        json.dump(metadata, f, indent=4)

    # 3. 保存预测结果
    for horizon in ["30min", "60min"]:
        pd.DataFrame(
            {
                "y_true": results[horizon]["y_true"].values,
                "y_pred": results[horizon]["y_pred"],
            }
        ).to_csv(f"{save_dir}/predictions_{horizon}.csv", index=False)

    # 4. 保存性能指标
    performance = {
        "30min": {
            "RMSE": float(results["30min"]["rmse"]),
            "MAE": float(results["30min"]["mae"]),
            "R2": float(results["30min"]["r2"]),
        },
        "60min": {
            "RMSE": float(results["60min"]["rmse"]),
            "MAE": float(results["60min"]["mae"]),
            "R2": float(results["60min"]["r2"]),
        },
    }
    with open(f"{save_dir}/performance.json", "w") as f:
        json.dump(performance, f, indent=4)

    logger.info(f"✓ Models and results saved to {save_dir}/")
    return timestamp


# 使用示例
if __name__ == "__main__":
    # 训练完成后
    save_models_simple(predictor, results, save_dir="saved_models")

In [ ]:
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)


class SimpleGlucosePredictor:
    """简洁版:加载和使用已保存的模型"""

    def __init__(self, model_dir="saved_models"):
        self.model_dir = model_dir

        # 加载模型
        self.model_30min = joblib.load(f"{model_dir}/model_30min.pkl")
        self.model_60min = joblib.load(f"{model_dir}/model_60min.pkl")

        # 加载元数据
        with open(f"{model_dir}/metadata.json", "r") as f:
            self.metadata = json.load(f)

        self.feature_names = self.metadata["feature_names"]
        logger.info(f"✓ Models loaded ({len(self.feature_names)} features)")

    def predict(self, X, horizon="both"):
        """
        进行预测

        Parameters:
        -----------
        X : pd.DataFrame - 输入特征
        horizon : str - "30min", "60min", 或 "both"

        Returns:
        --------
        dict - 预测结果
        """
        # 确保特征顺序正确
        X_ordered = X[self.feature_names]

        results = {}
        if horizon in ["30min", "both"]:
            results["30min"] = self.model_30min.predict(X_ordered)
        if horizon in ["60min", "both"]:
            results["60min"] = self.model_60min.predict(X_ordered)

        return results

    def get_performance(self):
        """获取模型性能指标"""
        with open(f"{self.model_dir}/performance.json", "r") as f:
            return json.load(f)


# ========== 使用示例 ==========

# 1. 加载模型
predictor = SimpleGlucosePredictor(model_dir="saved_models")

# 2. 查看性能
performance = predictor.get_performance()
print("模型性能:")
print(f"30分钟 - RMSE: {performance['30min']['RMSE']:.2f}")
print(f"60分钟 - RMSE: {performance['60min']['RMSE']:.2f}")

# 3. 在新数据上预测
new_data = pd.read_csv("test_enhanced_processed.csv")
exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]
X_new = new_data[[col for col in new_data.columns if col not in exclude_cols]]

# 进行预测
predictions = predictor.predict(X_new, horizon="both")

print("\n预测结果 (前5个样本):")
print(f"30分钟: {predictions['30min'][:5]}")
print(f"60分钟: {predictions['60min'][:5]}")

# 4. 对单个患者预测
# 对每个患者进行预测
for patient_id in new_data["patient_id"].unique()[:3]:  # 只看前3个患者
    patient_data = new_data[new_data["patient_id"] == patient_id]
    X_patient = patient_data[
        [col for col in patient_data.columns if col not in exclude_cols]
    ]

    if len(X_patient) > 0:
        patient_pred = predictor.predict(X_patient, horizon="30min")
        print(f"\n患者 {patient_id}:")
        print(f"  数据点数: {len(X_patient)}")
        print(f"  30分钟预测均值: {patient_pred['30min'].mean():.2f}")
        print(f"  前3个预测: {patient_pred['30min'][:3]}")


In [ ]:
import numpy as np
import pandas as pd


def analyze_patient_predictions(predictor, test_df, num_patients=5):
    """
    对每个患者进行详细的预测分析

    Parameters:
    -----------
    predictor : SimpleGlucosePredictor实例
    test_df : 测试数据
    num_patients : 分析的患者数量
    """
    exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]
    patient_ids = test_df["patient_id"].unique()[:num_patients]

    results_summary = []

    for patient_id in patient_ids:
        print(f"\n{'=' * 60}")
        print(f"患者 {patient_id} 的分析")
        print(f"{'=' * 60}")

        # 获取患者数据
        patient_data = test_df[test_df["patient_id"] == patient_id].copy()

        if len(patient_data) == 0:
            print(f"警告: 患者 {patient_id} 没有数据")
            continue

        # 准备特征
        X_patient = patient_data[
            [col for col in patient_data.columns if col not in exclude_cols]
        ]

        # 进行预测
        predictions = predictor.predict(X_patient, horizon="both")

        # 计算指标
        for horizon in ["30min", "60min"]:
            target_col = f"glucose_{horizon}"
            y_true = patient_data[target_col].values
            y_pred = predictions[horizon]

            # 移除NaN值
            valid_mask = ~np.isnan(y_true)
            y_true_clean = y_true[valid_mask]
            y_pred_clean = y_pred[valid_mask]

            if len(y_true_clean) > 0:
                rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
                mae = mean_absolute_error(y_true_clean, y_pred_clean)
                r2 = r2_score(y_true_clean, y_pred_clean)

                # 计算血糖范围统计
                hypoglycemia = np.sum(y_pred_clean < 70)  # 低血糖
                target_range = np.sum(
                    (y_pred_clean >= 70) & (y_pred_clean <= 180)
                )  # 目标范围
                hyperglycemia = np.sum(y_pred_clean > 180)  # 高血糖

                print(f"\n{horizon} 预测:")
                print(f"  数据点数: {len(y_true_clean)}")
                print(f"  RMSE: {rmse:.2f} mg/dL")
                print(f"  MAE: {mae:.2f} mg/dL")
                print(f"  R²: {r2:.4f}")
                print(f"  预测均值: {y_pred_clean.mean():.2f} mg/dL")
                print(
                    f"  预测范围: [{y_pred_clean.min():.2f}, {y_pred_clean.max():.2f}]"
                )
                print("\n  血糖范围分布:")
                print(
                    f"    低血糖 (<70): {hypoglycemia} ({hypoglycemia / len(y_pred_clean) * 100:.1f}%)"
                )
                print(
                    f"    目标范围 (70-180): {target_range} ({target_range / len(y_pred_clean) * 100:.1f}%)"
                )
                print(
                    f"    高血糖 (>180): {hyperglycemia} ({hyperglycemia / len(y_pred_clean) * 100:.1f}%)"
                )

                results_summary.append(
                    {
                        "patient_id": patient_id,
                        "horizon": horizon,
                        "n_samples": len(y_true_clean),
                        "RMSE": rmse,
                        "MAE": mae,
                        "R2": r2,
                        "mean_prediction": y_pred_clean.mean(),
                        "hypoglycemia_pct": hypoglycemia / len(y_pred_clean) * 100,
                        "target_range_pct": target_range / len(y_pred_clean) * 100,
                        "hyperglycemia_pct": hyperglycemia / len(y_pred_clean) * 100,
                    }
                )

    # 返回汇总DataFrame
    return pd.DataFrame(results_summary)


# 使用示例
summary_df = analyze_patient_predictions(predictor, new_data, num_patients=5)

# 保存结果
summary_df.to_csv("saved_models/patient_analysis.csv", index=False)
print(f"\n\n{'=' * 60}")
print("所有患者的汇总:")
print(f"{'=' * 60}")
print(summary_df)

# 按预测时间范围分组统计
print("\n\n按预测时间范围的整体统计:")
for horizon in ["30min", "60min"]:
    horizon_data = summary_df[summary_df["horizon"] == horizon]
    print(f"\n{horizon}:")
    print(
        f"  平均RMSE: {horizon_data['RMSE'].mean():.2f} ± {horizon_data['RMSE'].std():.2f}"
    )
    print(
        f"  平均MAE: {horizon_data['MAE'].mean():.2f} ± {horizon_data['MAE'].std():.2f}"
    )
    print(f"  平均R²: {horizon_data['R2'].mean():.4f} ± {horizon_data['R2'].std():.4f}")
    print(f"  目标范围内比例: {horizon_data['target_range_pct'].mean():.1f}%")


In [ ]:
def plot_patient_predictions(predictor, test_df, patient_id, n_points=600):
    """绘制单个患者的预测曲线"""
    import matplotlib.pyplot as plt

    exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]

    # 获取患者数据
    patient_data = test_df[test_df["patient_id"] == patient_id].copy()
    X_patient = patient_data[
        [col for col in patient_data.columns if col not in exclude_cols]
    ]

    # 预测
    predictions = predictor.predict(X_patient, horizon="both")

    # 截取前n_points个点
    n = min(n_points, len(patient_data))

    fig, axes = plt.subplots(2, 1, figsize=(18, 10), dpi=300)

    for idx, horizon in enumerate(["30min", "60min"]):
        y_true = patient_data[f"glucose_{horizon}"].values[:n]
        y_pred = predictions[horizon][:n]

        # 移除NaN
        valid_mask = ~np.isnan(y_true)
        time_indices = np.arange(n)[valid_mask]
        y_true = y_true[valid_mask]
        y_pred = y_pred[valid_mask]

        # 绘图
        axes[idx].plot(time_indices, y_true, "b-", label="True", linewidth=2, alpha=0.7)
        axes[idx].plot(
            time_indices, y_pred, "r--", label="Predict", linewidth=2, alpha=0.7
        )
        axes[idx].fill_between(time_indices, y_true, y_pred, alpha=0.2, color="gray")

        # 添加血糖范围
        axes[idx].axhspan(70, 180, alpha=0.1, color="green", label="Target")
        axes[idx].axhline(y=70, color="orange", linestyle=":", alpha=0.5)
        axes[idx].axhline(y=180, color="orange", linestyle=":", alpha=0.5)

        # 计算指标
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)

        axes[idx].set_title(
            f"Patient {patient_id} - {horizon} Prediction (RMSE: {rmse:.2f}, MAE: {mae:.2f})",
            fontsize=14,
            fontweight="bold",
        )
        axes[idx].set_xlabel("Time point (5min)", fontsize=12)
        axes[idx].set_ylabel("Glucose (mg/dL)", fontsize=12)
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        f"saved_models/patient_{patient_id}_predictions.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


# 使用示例:绘制患者559的预测
plot_patient_predictions(predictor, new_data, patient_id=559, n_points=600)

In [ ]:
def plot_patient_predictions_enhanced(
    predictor, test_df, patient_id, n_points=600, save_dir="saved_models"
):
    """
    绘制单个患者的预测曲线(增强版,包含详细性能指标)

    Parameters:
    -----------
    predictor : SimpleGlucosePredictor实例
    test_df : 测试数据
    patient_id : 患者ID
    n_points : 绘制的时间点数量
    save_dir : 保存目录
    """
    exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]

    # 获取患者数据
    patient_data = test_df[test_df["patient_id"] == patient_id].copy()

    if len(patient_data) == 0:
        logger.warning(f"患者 {patient_id} 没有数据")
        return None

    X_patient = patient_data[
        [col for col in patient_data.columns if col not in exclude_cols]
    ]

    if len(X_patient) == 0:
        logger.warning(f"患者 {patient_id} 特征数据为空")
        return None

    # 进行预测
    predictions = predictor.predict(X_patient, horizon="both")

    # 创建图表
    fig, axes = plt.subplots(2, 1, figsize=(18, 10), dpi=300)

    metrics_summary = {}

    for idx, horizon in enumerate(["30min", "60min"]):
        # 获取真实值和预测值
        y_true = patient_data[f"glucose_{horizon}"].values
        y_pred = predictions[horizon]

        # 对齐长度
        min_len = min(len(y_true), len(y_pred))
        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]

        # 截取前n_points个点
        n = min(n_points, len(y_true))
        y_true_plot = y_true[:n]
        y_pred_plot = y_pred[:n]

        # 移除NaN值
        valid_mask = ~np.isnan(y_true_plot)
        time_indices = np.arange(n)[valid_mask]
        y_true_clean = y_true_plot[valid_mask]
        y_pred_clean = y_pred_plot[valid_mask]

        if len(y_true_clean) == 0:
            logger.warning(f"患者 {patient_id} 的 {horizon} 数据全部为NaN")
            continue

        # 计算性能指标
        rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
        mae = mean_absolute_error(y_true_clean, y_pred_clean)
        r2 = r2_score(y_true_clean, y_pred_clean)

        # 保存指标
        metrics_summary[horizon] = {
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "n_samples": len(y_true_clean),
        }

        # 绘制预测曲线
        axes[idx].plot(
            time_indices,
            y_true_clean,
            "b-",
            label="True Glucose",
            linewidth=2,
            alpha=0.7,
        )
        axes[idx].plot(
            time_indices,
            y_pred_clean,
            "r--",
            label="Predicted Glucose",
            linewidth=2,
            alpha=0.7,
        )
        axes[idx].fill_between(
            time_indices, y_true_clean, y_pred_clean, alpha=0.2, color="gray"
        )

        # 添加血糖目标范围
        axes[idx].axhspan(
            70, 180, alpha=0.1, color="green", label="Target Range (70-180 mg/dL)"
        )
        axes[idx].axhline(y=70, color="orange", linestyle=":", alpha=0.5)
        axes[idx].axhline(y=180, color="orange", linestyle=":", alpha=0.5)

        # 设置标题(包含详细性能指标)
        axes[idx].set_title(
            f"Test Patient {patient_id} - {horizon} Prediction\n"
            f"RMSE: {rmse:.2f} mg/dL, MAE: {mae:.2f} mg/dL, R²: {r2:.4f} (n={len(y_true_clean)})",
            fontsize=14,
            fontweight="bold",
            pad=15,
        )

        axes[idx].set_xlabel("Time Point (5-min intervals)", fontsize=12)
        axes[idx].set_ylabel("Glucose (mg/dL)", fontsize=12)
        axes[idx].legend(fontsize=10, loc="best")
        axes[idx].grid(True, alpha=0.3)

        # 设置Y轴范围
        y_min = min(y_true_clean.min(), y_pred_clean.min()) - 20
        y_max = max(y_true_clean.max(), y_pred_clean.max()) + 20
        axes[idx].set_ylim([y_min, y_max])

    plt.tight_layout()

    # 保存图片
    save_path = f"{save_dir}/test_patient_{patient_id}_predictions.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 打印性能指标
    print(f"\n{'=' * 70}")
    print(f"Test Patient {patient_id} - Performance Metrics")
    print(f"{'=' * 70}")
    for horizon in ["30min", "60min"]:
        if horizon in metrics_summary:
            metrics = metrics_summary[horizon]
            print(f"\n{horizon} Prediction:")
            print(f"  RMSE: {metrics['RMSE']:.2f} mg/dL")
            print(f"  MAE:  {metrics['MAE']:.2f} mg/dL")
            print(f"  R²:   {metrics['R2']:.4f}")
            print(f"  Samples: {metrics['n_samples']}")
    print(f"{'=' * 70}")

    logger.info(f"✓ Test patient {patient_id} predictions saved to {save_path}")

    return metrics_summary


def evaluate_all_test_patients(
    predictor, test_df, test_patient_ids, n_points=600, save_dir="saved_models"
):
    """
    评估所有测试集患者并生成汇总报告

    Parameters:
    -----------
    predictor : SimpleGlucosePredictor实例
    test_df : 测试数据
    test_patient_ids : 测试集患者ID列表(如[588, 591])
    n_points : 绘制的时间点数量
    save_dir : 保存目录
    """
    all_metrics = []

    print(f"\n{'=' * 70}")
    print(f"Evaluating {len(test_patient_ids)} Test Patients")
    print(f"{'=' * 70}")

    for patient_id in test_patient_ids:
        print(f"\nProcessing Patient {patient_id}...")

        # 绘制并获取指标
        metrics = plot_patient_predictions_enhanced(
            predictor, test_df, patient_id, n_points=n_points, save_dir=save_dir
        )

        if metrics:
            for horizon in ["30min", "60min"]:
                if horizon in metrics:
                    all_metrics.append(
                        {
                            "patient_id": patient_id,
                            "horizon": horizon,
                            "RMSE": metrics[horizon]["RMSE"],
                            "MAE": metrics[horizon]["MAE"],
                            "R2": metrics[horizon]["R2"],
                            "n_samples": metrics[horizon]["n_samples"],
                        }
                    )

    # 创建汇总DataFrame
    summary_df = pd.DataFrame(all_metrics)

    # 保存汇总结果
    summary_path = f"{save_dir}/test_patients_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    # 打印汇总统计
    print(f"\n{'=' * 70}")
    print("Overall Test Set Performance (Average across all test patients)")
    print(f"{'=' * 70}")

    for horizon in ["30min", "60min"]:
        horizon_data = summary_df[summary_df["horizon"] == horizon]
        if len(horizon_data) > 0:
            print(f"\n{horizon} Prediction:")
            print(
                f"  Average RMSE: {horizon_data['RMSE'].mean():.2f} ± {horizon_data['RMSE'].std():.2f} mg/dL"
            )
            print(
                f"  Average MAE:  {horizon_data['MAE'].mean():.2f} ± {horizon_data['MAE'].std():.2f} mg/dL"
            )
            print(
                f"  Average R²:   {horizon_data['R2'].mean():.4f} ± {horizon_data['R2'].std():.4f}"
            )
            print(f"  Total Samples: {int(horizon_data['n_samples'].sum())}")

    print(f"\n{'=' * 70}")
    print(f"Summary saved to: {summary_path}")
    print(f"{'=' * 70}\n")

    return summary_df


# ========== 使用示例 ==========

# 方法1: 绘制单个测试患者
plot_patient_predictions_enhanced(
    predictor,
    new_data,  # 或 test_df
    patient_id=588,
    n_points=600,
    save_dir="saved_models",
)

In [ ]:
# 方法2: 评估所有测试患者(假设测试集包含患者588和591)
test_patient_ids = [559, 563, 570, 575, 588, 591]
summary_df = evaluate_all_test_patients(
    predictor,
    new_data,  # 或 test_df
    test_patient_ids=test_patient_ids,
    n_points=600,
    save_dir="saved_models",
)

# 查看汇总结果
print("\nTest Patients Summary:")
print(summary_df)

In [ ]:
import numpy as np
import pandas as pd
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def calculate_overall_test_metrics(predictor, test_df, save_dir="saved_models"):
    """
    计算整个测试集的性能指标(所有患者混合)

    Parameters:
    -----------
    predictor : SimpleGlucosePredictor实例
    test_df : 测试数据集
    save_dir : 保存目录

    Returns:
    --------
    dict : 包含各项指标的字典
    """
    exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]

    print(f"\n{'=' * 80}")
    print("Computing Test Metrics on Entire Test Set")
    print(f"{'=' * 80}")

    # 准备特征数据
    X_test = test_df[[col for col in test_df.columns if col not in exclude_cols]]

    if len(X_test) == 0:
        logger.error("Test data is empty")
        return {}

    # 进行预测
    predictions = predictor.predict(X_test, horizon="both")

    all_metrics = {}

    # 对每个预测时间范围计算指标
    for horizon in ["30min", "60min"]:
        y_true = test_df[f"glucose_{horizon}"].values
        y_pred = predictions[horizon]

        # 对齐长度
        min_len = min(len(y_true), len(y_pred))
        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]

        # 移除NaN值
        valid_mask = ~np.isnan(y_true)
        y_true_clean = y_true[valid_mask]
        y_pred_clean = y_pred[valid_mask]

        if len(y_true_clean) == 0:
            logger.warning(f"{horizon}: No valid samples")
            continue

        # 计算性能指标
        rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
        mae = mean_absolute_error(y_true_clean, y_pred_clean)
        r2 = r2_score(y_true_clean, y_pred_clean)

        # 计算残差统计
        residuals = y_true_clean - y_pred_clean
        mean_residual = np.mean(residuals)
        std_residual = np.std(residuals)
        median_residual = np.median(residuals)

        # 血糖范围统计
        hypoglycemia_count = np.sum(y_pred_clean < 70)
        target_range_count = np.sum((y_pred_clean >= 70) & (y_pred_clean <= 180))
        hyperglycemia_count = np.sum(y_pred_clean > 180)

        total_samples = len(y_pred_clean)
        hypoglycemia_pct = (hypoglycemia_count / total_samples) * 100
        target_range_pct = (target_range_count / total_samples) * 100
        hyperglycemia_pct = (hyperglycemia_count / total_samples) * 100

        # 计算真实值的血糖范围(用于对比)
        true_hypoglycemia = np.sum(y_true_clean < 70)
        true_target = np.sum((y_true_clean >= 70) & (y_true_clean <= 180))
        true_hyperglycemia = np.sum(y_true_clean > 180)

        # 保存指标
        all_metrics[horizon] = {
            "n_samples": total_samples,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "Mean_Residual": mean_residual,
            "Std_Residual": std_residual,
            "Median_Residual": median_residual,
            "Mean_True": y_true_clean.mean(),
            "Std_True": y_true_clean.std(),
            "Mean_Pred": y_pred_clean.mean(),
            "Std_Pred": y_pred_clean.std(),
            "Hypoglycemia_Count": hypoglycemia_count,
            "Hypoglycemia_%": hypoglycemia_pct,
            "Target_Range_Count": target_range_count,
            "Target_Range_%": target_range_pct,
            "Hyperglycemia_Count": hyperglycemia_count,
            "Hyperglycemia_%": hyperglycemia_pct,
            "True_Hypoglycemia_%": (true_hypoglycemia / total_samples) * 100,
            "True_Target_%": (true_target / total_samples) * 100,
            "True_Hyperglycemia_%": (true_hyperglycemia / total_samples) * 100,
        }

        # 打印结果
        print(f"\n{horizon} Prediction:")
        print(f"  Total Samples: {total_samples:,}")
        print(f"  RMSE: {rmse:.2f} mg/dL")
        print(f"  MAE:  {mae:.2f} mg/dL")
        print(f"  R²:   {r2:.4f}")
        print(f"  Mean Residual: {mean_residual:.2f} mg/dL")
        print(f"  Std Residual:  {std_residual:.2f} mg/dL")
        print(f"  Median Residual: {median_residual:.2f} mg/dL")
        print("\n  Glucose Statistics:")
        print(
            f"    True Values:  {y_true_clean.mean():.2f} ± {y_true_clean.std():.2f} mg/dL"
        )
        print(
            f"    Predictions:  {y_pred_clean.mean():.2f} ± {y_pred_clean.std():.2f} mg/dL"
        )
        print("\n  Glycemic Range Distribution (Predicted):")
        print(
            f"    Hypoglycemia (<70 mg/dL):     {hypoglycemia_count:5d} ({hypoglycemia_pct:5.1f}%)"
        )
        print(
            f"    Target Range (70-180 mg/dL):  {target_range_count:5d} ({target_range_pct:5.1f}%)"
        )
        print(
            f"    Hyperglycemia (>180 mg/dL):   {hyperglycemia_count:5d} ({hyperglycemia_pct:5.1f}%)"
        )
        print("\n  Glycemic Range Distribution (True):")
        print(
            f"    Hypoglycemia:   {true_hypoglycemia:5d} ({(true_hypoglycemia / total_samples) * 100:5.1f}%)"
        )
        print(
            f"    Target Range:   {true_target:5d} ({(true_target / total_samples) * 100:5.1f}%)"
        )
        print(
            f"    Hyperglycemia:  {true_hyperglycemia:5d} ({(true_hyperglycemia / total_samples) * 100:5.1f}%)"
        )

    print(f"\n{'=' * 80}\n")

    # 保存结果到CSV
    metrics_records = []
    for horizon, metrics in all_metrics.items():
        record = {"Horizon": horizon}
        record.update(metrics)
        metrics_records.append(record)

    metrics_df = pd.DataFrame(metrics_records)
    save_path = f"{save_dir}/test_set_overall_metrics.csv"
    metrics_df.to_csv(save_path, index=False)
    logger.info(f"Overall test metrics saved to: {save_path}")

    return all_metrics


def calculate_per_patient_test_metrics(predictor, test_df, save_dir="saved_models"):
    """
    计算每个患者在测试集上的性能指标(带均值和标准差)

    Parameters:
    -----------
    predictor : SimpleGlucosePredictor实例
    test_df : 测试数据集
    save_dir : 保存目录

    Returns:
    --------
    pd.DataFrame : 每个患者的详细指标
    dict : 汇总统计(均值±标准差)
    """
    exclude_cols = ["patient_id", "timestamp", "glucose_30min", "glucose_60min"]

    patient_ids = sorted(test_df["patient_id"].unique())

    print(f"\n{'=' * 80}")
    print(f"Computing Per-Patient Test Metrics ({len(patient_ids)} patients)")
    print(f"{'=' * 80}")

    all_metrics = []

    # 遍历每个患者
    for patient_id in patient_ids:
        patient_data = test_df[test_df["patient_id"] == patient_id].copy()

        if len(patient_data) == 0:
            continue

        X_patient = patient_data[
            [col for col in patient_data.columns if col not in exclude_cols]
        ]

        if len(X_patient) == 0:
            continue

        try:
            # 进行预测
            predictions = predictor.predict(X_patient, horizon="both")

            for horizon in ["30min", "60min"]:
                y_true = patient_data[f"glucose_{horizon}"].values
                y_pred = predictions[horizon]

                # 对齐长度
                min_len = min(len(y_true), len(y_pred))
                y_true = y_true[:min_len]
                y_pred = y_pred[:min_len]

                # 移除NaN
                valid_mask = ~np.isnan(y_true)
                y_true_clean = y_true[valid_mask]
                y_pred_clean = y_pred[valid_mask]

                if len(y_true_clean) < 5:
                    continue

                # 计算指标
                rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
                mae = mean_absolute_error(y_true_clean, y_pred_clean)
                r2 = r2_score(y_true_clean, y_pred_clean)

                # 残差统计
                residuals = y_true_clean - y_pred_clean
                mean_residual = np.mean(residuals)
                std_residual = np.std(residuals)

                # 血糖范围
                hypoglycemia_count = np.sum(y_pred_clean < 70)
                target_count = np.sum((y_pred_clean >= 70) & (y_pred_clean <= 180))
                hyperglycemia_count = np.sum(y_pred_clean > 180)

                hypoglycemia_pct = (hypoglycemia_count / len(y_pred_clean)) * 100
                target_pct = (target_count / len(y_pred_clean)) * 100
                hyperglycemia_pct = (hyperglycemia_count / len(y_pred_clean)) * 100

                all_metrics.append(
                    {
                        "patient_id": patient_id,
                        "horizon": horizon,
                        "n_samples": len(y_true_clean),
                        "RMSE": rmse,
                        "MAE": mae,
                        "R2": r2,
                        "Mean_Residual": mean_residual,
                        "Std_Residual": std_residual,
                        "Hypoglycemia_%": hypoglycemia_pct,
                        "Target_Range_%": target_pct,
                        "Hyperglycemia_%": hyperglycemia_pct,
                    }
                )

                logger.info(
                    f"Patient {patient_id} ({horizon}): "
                    f"RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}, n={len(y_true_clean)}"
                )

        except Exception as e:
            logger.error(f"Patient {patient_id}: Error - {e}")
            continue

    # 创建DataFrame
    metrics_df = pd.DataFrame(all_metrics)

    if len(metrics_df) == 0:
        logger.error("No valid metrics computed")
        return pd.DataFrame(), {}

    # 计算汇总统计
    summary_stats = {}

    print(f"\n{'=' * 80}")
    print("SUMMARY STATISTICS (Mean ± Std) ACROSS ALL PATIENTS")
    print(f"{'=' * 80}")

    for horizon in ["30min", "60min"]:
        horizon_data = metrics_df[metrics_df["horizon"] == horizon]

        if len(horizon_data) == 0:
            continue

        stats = {
            "n_patients": len(horizon_data),
            "total_samples": int(horizon_data["n_samples"].sum()),
            "RMSE_mean": horizon_data["RMSE"].mean(),
            "RMSE_std": horizon_data["RMSE"].std(),
            "RMSE_min": horizon_data["RMSE"].min(),
            "RMSE_max": horizon_data["RMSE"].max(),
            "MAE_mean": horizon_data["MAE"].mean(),
            "MAE_std": horizon_data["MAE"].std(),
            "MAE_min": horizon_data["MAE"].min(),
            "MAE_max": horizon_data["MAE"].max(),
            "R2_mean": horizon_data["R2"].mean(),
            "R2_std": horizon_data["R2"].std(),
            "R2_min": horizon_data["R2"].min(),
            "R2_max": horizon_data["R2"].max(),
            "Mean_Residual_mean": horizon_data["Mean_Residual"].mean(),
            "Mean_Residual_std": horizon_data["Mean_Residual"].std(),
            "Std_Residual_mean": horizon_data["Std_Residual"].mean(),
            "Std_Residual_std": horizon_data["Std_Residual"].std(),
            "Hypoglycemia_mean": horizon_data["Hypoglycemia_%"].mean(),
            "Hypoglycemia_std": horizon_data["Hypoglycemia_%"].std(),
            "Target_Range_mean": horizon_data["Target_Range_%"].mean(),
            "Target_Range_std": horizon_data["Target_Range_%"].std(),
            "Hyperglycemia_mean": horizon_data["Hyperglycemia_%"].mean(),
            "Hyperglycemia_std": horizon_data["Hyperglycemia_%"].std(),
        }

        summary_stats[horizon] = stats

        # 打印格式化的结果
        print(f"\n{horizon} Prediction:")
        print(f"  Number of Patients: {stats['n_patients']}")
        print(f"  Total Samples: {stats['total_samples']:,}")
        print("\n  Performance Metrics (Mean ± Std):")
        print(
            f"    RMSE: {stats['RMSE_mean']:.2f} ± {stats['RMSE_std']:.2f} mg/dL  [range: {stats['RMSE_min']:.2f} - {stats['RMSE_max']:.2f}]"
        )
        print(
            f"    MAE:  {stats['MAE_mean']:.2f} ± {stats['MAE_std']:.2f} mg/dL  [range: {stats['MAE_min']:.2f} - {stats['MAE_max']:.2f}]"
        )
        print(
            f"    R²:   {stats['R2_mean']:.4f} ± {stats['R2_std']:.4f}  [range: {stats['R2_min']:.4f} - {stats['R2_max']:.4f}]"
        )
        print("\n  Residual Statistics (Mean ± Std):")
        print(
            f"    Mean Residual: {stats['Mean_Residual_mean']:.2f} ± {stats['Mean_Residual_std']:.2f} mg/dL"
        )
        print(
            f"    Std Residual:  {stats['Std_Residual_mean']:.2f} ± {stats['Std_Residual_std']:.2f} mg/dL"
        )
        print("\n  Glycemic Range Distribution (Mean ± Std):")
        print(
            f"    Hypoglycemia (<70):     {stats['Hypoglycemia_mean']:.1f} ± {stats['Hypoglycemia_std']:.1f} %"
        )
        print(
            f"    Target Range (70-180):  {stats['Target_Range_mean']:.1f} ± {stats['Target_Range_std']:.1f} %"
        )
        print(
            f"    Hyperglycemia (>180):   {stats['Hyperglycemia_mean']:.1f} ± {stats['Hyperglycemia_std']:.1f} %"
        )

    print(f"{'=' * 80}\n")

    # 保存文件
    detailed_path = f"{save_dir}/per_patient_test_metrics.csv"
    metrics_df.to_csv(detailed_path, index=False)
    logger.info(f"Per-patient metrics saved to: {detailed_path}")

    # 保存汇总统计
    summary_records = []
    for horizon, stats in summary_stats.items():
        summary_records.append(
            {
                "Horizon": horizon,
                "N_Patients": stats["n_patients"],
                "Total_Samples": stats["total_samples"],
                "RMSE_Mean": stats["RMSE_mean"],
                "RMSE_Std": stats["RMSE_std"],
                "RMSE_Min": stats["RMSE_min"],
                "RMSE_Max": stats["RMSE_max"],
                "MAE_Mean": stats["MAE_mean"],
                "MAE_Std": stats["MAE_std"],
                "MAE_Min": stats["MAE_min"],
                "MAE_Max": stats["MAE_max"],
                "R2_Mean": stats["R2_mean"],
                "R2_Std": stats["R2_std"],
                "R2_Min": stats["R2_min"],
                "R2_Max": stats["R2_max"],
                "Mean_Residual_Mean": stats["Mean_Residual_mean"],
                "Mean_Residual_Std": stats["Mean_Residual_std"],
                "Std_Residual_Mean": stats["Std_Residual_mean"],
                "Std_Residual_Std": stats["Std_Residual_std"],
                "Hypoglycemia_Mean": stats["Hypoglycemia_mean"],
                "Hypoglycemia_Std": stats["Hypoglycemia_std"],
                "Target_Range_Mean": stats["Target_Range_mean"],
                "Target_Range_Std": stats["Target_Range_std"],
                "Hyperglycemia_Mean": stats["Hyperglycemia_mean"],
                "Hyperglycemia_Std": stats["Hyperglycemia_std"],
            }
        )

    summary_df = pd.DataFrame(summary_records)
    summary_path = f"{save_dir}/per_patient_summary_statistics.csv"
    summary_df.to_csv(summary_path, index=False)
    logger.info(f"Summary statistics saved to: {summary_path}")

    return metrics_df, summary_stats


# ========== 主程序 ==========
if __name__ == "__main__":
    # 1. 加载模型
    logger.info("Loading model...")
    predictor = SimpleGlucosePredictor(model_dir="saved_models")

    # 2. 加载测试数据
    test_df = pd.read_csv("test_enhanced_processed.csv")
    logger.info(
        f"Test set: {len(test_df)} samples, {len(test_df['patient_id'].unique())} patients"
    )

    # 3. 计算整体测试集指标
    overall_metrics = calculate_overall_test_metrics(
        predictor=predictor, test_df=test_df, save_dir="saved_models"
    )

    # 4. 计算每个患者的指标(带均值±标准差)
    per_patient_metrics, summary_stats = calculate_per_patient_test_metrics(
        predictor=predictor, test_df=test_df, save_dir="saved_models"
    )

    # 5. 查看每个患者的详细指标
    print("\nPer-Patient Detailed Metrics:")
    print(per_patient_metrics.to_string(index=False))

    print("\n✓ All metrics computed and saved!")


In [ ]:
import numpy as np
import pandas as pd

# 从日志中提取的完整测试集数据
test_data = {
    "patient_id": [559, 563, 570, 575, 588, 591] * 2,
    "horizon": ["30min"] * 6 + ["60min"] * 6,
    "n_samples": [
        2508,
        2562,
        2753,
        2604,
        2779,
        2753,
        2508,
        2562,
        2753,
        2604,
        2779,
        2753,
    ],
    "RMSE": [
        19.97,
        18.99,
        19.02,
        23.61,
        18.18,
        22.09,
        33.68,
        30.86,
        30.33,
        36.22,
        30.84,
        33.47,
    ],
    "MAE": [
        13.88,
        13.47,
        12.21,
        15.55,
        13.32,
        15.99,
        24.39,
        22.41,
        21.27,
        26.40,
        22.64,
        25.82,
    ],
    "R2": [
        0.9113,
        0.8306,
        0.9178,
        0.8481,
        0.8583,
        0.8137,
        0.7485,
        0.5520,
        0.7899,
        0.6423,
        0.5985,
        0.5697,
    ],
}

df = pd.DataFrame(test_data)

print("=" * 80)
print("测试集性能统计 (所有6个患者的测试数据)")
print("=" * 80)

# 分别计算30min和60min的统计量
summary_results = []

for horizon in ["30min", "60min"]:
    horizon_data = df[df["horizon"] == horizon]

    n_patients = len(horizon_data)
    total_samples = int(horizon_data["n_samples"].sum())

    rmse_mean = horizon_data["RMSE"].mean()
    rmse_std = horizon_data["RMSE"].std(ddof=1)
    rmse_min = horizon_data["RMSE"].min()
    rmse_max = horizon_data["RMSE"].max()

    mae_mean = horizon_data["MAE"].mean()
    mae_std = horizon_data["MAE"].std(ddof=1)
    mae_min = horizon_data["MAE"].min()
    mae_max = horizon_data["MAE"].max()

    r2_mean = horizon_data["R2"].mean()
    r2_std = horizon_data["R2"].std(ddof=1)
    r2_min = horizon_data["R2"].min()
    r2_max = horizon_data["R2"].max()

    print(f"\n{horizon} 预测 (基于{n_patients}个患者):")
    print(f"  总样本数: {total_samples:,}")
    print("\n  性能指标 (均值 ± 标准差):")
    print(
        f"    RMSE: {rmse_mean:.4f} ± {rmse_std:.4f} mg/dL  [范围: {rmse_min:.4f} - {rmse_max:.4f}]"
    )
    print(
        f"    MAE:  {mae_mean:.4f} ± {mae_std:.4f} mg/dL  [范围: {mae_min:.4f} - {mae_max:.4f}]"
    )
    print(
        f"    R²:   {r2_mean:.4f} ± {r2_std:.4f}  [范围: {r2_min:.4f} - {r2_max:.4f}]"
    )

    print("\n  各患者详细数据:")
    for _, row in horizon_data.iterrows():
        print(
            f"    患者{int(row['patient_id'])}: RMSE={row['RMSE']:.4f}, MAE={row['MAE']:.4f}, R²={row['R2']:.4f}, n={int(row['n_samples'])}"
        )

    summary_results.append(
        {
            "Horizon": horizon,
            "N_Patients": n_patients,
            "Total_Samples": total_samples,
            "RMSE_Mean": rmse_mean,
            "RMSE_Std": rmse_std,
            "RMSE_Min": rmse_min,
            "RMSE_Max": rmse_max,
            "MAE_Mean": mae_mean,
            "MAE_Std": mae_std,
            "MAE_Min": mae_min,
            "MAE_Max": mae_max,
            "R2_Mean": r2_mean,
            "R2_Std": r2_std,
            "R2_Min": r2_min,
            "R2_Max": r2_max,
        }
    )

print("=" * 80)

# 创建汇总表
summary_df = pd.DataFrame(summary_results)
summary_df